# Práctica de Model Context Protocol (MCP)

Esta práctica consta de dos partes principales:
1. **Parte 1:** Crear tu propio cliente MCP en Python y conectarlo a un servidor local/público usando transporte stdio.
2. **Parte 2:** Crear un servidor MCP usando `FastMCP` y desplegarlo en una **Azure Function** usando el modelo de programación v2 de Python y el transporte HTTP SSE (Server-Sent Events).

---

## ¿Qué es Model Context Protocol (MCP)?

El **Model Context Protocol (MCP)** es un estándar abierto desarrollado para permitir que los modelos de lenguaje (LLMs) se conecten de manera segura y bidireccional a fuentes de datos y herramientas externas. 

Funciona con un modelo Cliente-Servidor:
- **MCP Server:** Expone herramientas (tools), recursos (resources) y prompts.
- **MCP Client:** Se conecta al servidor, descubre qué herramientas están disponibles y las ejecuta en nombre del LLM.

Soporta dos métodos principales de transporte:
1. **stdio:** El cliente arranca el servidor como un subproceso y se comunica mediante entrada/salida estándar (estándar de facto para integraciones locales como Claude Desktop).
2. **SSE (Server-Sent Events):** El cliente se conecta al servidor a través de HTTP. El servidor envía eventos en tiempo real (SSE) y el cliente responde enviando solicitudes HTTP POST.

## Parte 1: Crear tu propio cliente MCP

Para probar el cliente, hemos creado un servidor MCP simple llamado `math_server.py` que ofrece herramientas matemáticas básicas (`add`, `multiply` y `get_system_info`).

El código del servidor está en [math_server.py](./math_server.py). Veamos cómo conectarnos a él e interactuar con sus herramientas directamente desde el código usando el SDK de MCP.

In [ ]:
# Importar las clases necesarias del SDK de MCP
import sys
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Configurar los parámetros para arrancar el servidor math_server.py en un subproceso
python_executable = sys.executable
server_script = "math_server.py"

server_params = StdioServerParameters(
    command=python_executable,
    args=[server_script],
    env=None
)

print(f"Parámetros de conexión: {python_executable} {server_script}")

### Ejecutar el Cliente de forma asíncrona

Como Jupyter Notebook ya corre en un event loop de `asyncio`, podemos utilizar `await` directamente sin necesidad de usar `asyncio.run()`. Vamos a inicializar la sesión, listar las herramientas y ejecutar una de ellas.

In [ ]:
# Conectarse usando el cliente stdio
async with stdio_client(server_params) as (read_stream, write_stream):
    # Crear la sesión de cliente
    async with ClientSession(read_stream, write_stream) as session:
        # 1. Inicializar la comunicación
        print("Inicializando conexión con el servidor...")
        await session.initialize()
        print("¡Servidor MCP inicializado con éxito!")
        
        # 2. Listar las herramientas disponibles
        tools_response = await session.list_tools()
        print("\n--- Herramientas Disponibles en el Servidor ---")
        for tool in tools_response.tools:
            print(f"- Nombre: {tool.name}")
            print(f"  Descripción: {tool.description}")
            print(f"  Esquema de entrada: {tool.inputSchema}\n")
            
        # 3. Invocar la herramienta 'add'
        val_a, val_b = 45.3, 12.7
        print(f"Invocando 'add' con a={val_a}, b={val_b}...")
        result_add = await session.call_tool("add", {"a": val_a, "b": val_b})
        print(f"Resultado de la suma: {result_add.content[0].text}")
        
        # 4. Invocar la herramienta 'get_system_info'
        print("\nInvocando 'get_system_info'...")
        result_sys = await session.call_tool("get_system_info", {})
        print(f"Información del sistema: {result_sys.content[0].text}")

--- 
## Parte 2: Crear un servidor MCP y desplegarlo en Azure Functions

Para exponer un servidor MCP sobre HTTP para clientes remotos, usamos el protocolo **SSE (Server-Sent Events)**.

Una excelente manera serverless de hospedar esto en la nube de Azure es mediante **Azure Functions (Python Programming Model v2)** utilizando la clase `func.AsgiFunctionApp` para envolver una aplicación **FastAPI**.

### Estructura del Proyecto de la Azure Function
La Azure Function se encuentra en la carpeta `./azure_function/` y está estructurada de la siguiente manera:
- `host.json`: Configura el runtime de Azure Functions.
- `local.settings.json`: Configura variables locales (almacenamiento de desarrollo, variables de entorno).
- `requirements.txt`: Dependencias de la Function (`azure-functions`, `mcp`, `fastapi`, `uvicorn`, `anyio`).
- `function_app.py`: La definición del servidor FastAPI, el servidor FastMCP, sus herramientas, y la envoltura ASGI de Azure Functions.

Puedes revisar el código de `function_app.py` haciendo clic aquí: [function_app.py](./azure_function/function_app.py).

### ¿Cómo funciona el servidor en `function_app.py`?

1. Inicializamos una aplicación FastAPI standard:
   ```python
   fastapi_app = FastAPI(title="MCP Server")
   ```
2. Creamos un servidor MCP con `FastMCP` de la librería `mcp`:
   ```python
   mcp = FastMCP("AzureFunctionsMCPServer")
   ```
3. Definimos herramientas en nuestro servidor usando `@mcp.tool()` (por ejemplo, para gestionar una base de datos de tareas en memoria).
4. Montamos la aplicación SSE de FastMCP en FastAPI en la ruta `/mcp`:
   ```python
   fastapi_app.mount("/mcp", mcp.sse_app())
   ```
   Esto crea automáticamente dos endpoints en FastAPI:
   - `GET /mcp/sse`: Donde el cliente se conecta para recibir la corriente de eventos (SSE).
   - `POST /mcp/messages`: Donde el cliente envía mensajes HTTP POST con peticiones JSON-RPC.
5. Envolvemos la app de FastAPI en un `func.AsgiFunctionApp` para Azure Functions:
   ```python
   app = func.AsgiFunctionApp(app=fastapi_app, http_auth_level=func.AuthLevel.ANONYMOUS)
   ```

### Probando la Azure Function Localmente

Para probar la Azure Function localmente:
1. Abre una terminal en tu máquina.
2. Ve al directorio de la Azure Function:
   ```bash
   cd notebooks/Desarrollo/Agentes/mcp/azure_function
   ```
3. Asegúrate de tener instalado **Azure Functions Core Tools** e inicia la función:
   ```bash
   func start
   ```
Esto levantará la función en `http://localhost:7071`. Por lo tanto, el endpoint de MCP SSE estará expuesto en:
- Conexión SSE: `http://localhost:7071/mcp/sse`

---

### Conectando un cliente SSE a la Azure Function

Una vez que la Azure Function esté corriendo en segundo plano (`func start`), puedes ejecutar la siguiente celda de código para probar la conexión cliente-servidor a través de SSE.

In [ ]:
from mcp.client.sse import sse_client
from mcp import ClientSession
import httpx

url = "http://localhost:7071/mcp/sse"
print(f"Intentando conectar al servidor MCP SSE en {url}...")

try:
    # Conectarse al cliente SSE de MCP
    async with sse_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            # Inicializar
            await session.initialize()
            print("¡Conectado e Inicializado con éxito con la Azure Function!")
            
            # Listar herramientas de la Azure Function
            tools_resp = await session.list_tools()
            print("\n--- Herramientas del Servidor Azure Function ---")
            for t in tools_resp.tools:
                print(f"- {t.name}: {t.description}")
            
            # Invocar 'list_tasks'
            print("\nInvocando 'list_tasks'...")
            tasks_res = await session.call_tool("list_tasks", {})
            print(f"Resultado del servidor:\n{tasks_res.content[0].text}")
            
            # Añadir una tarea
            nueva_tarea = "Completar la práctica de IAG con éxito"
            print(f"\nInvocando 'add_task' para añadir: '{nueva_tarea}'...")
            add_res = await session.call_tool("add_task", {"title": nueva_tarea})
            print(f"Resultado del servidor: {add_res.content[0].text}")
            
            # Listar de nuevo para ver la tarea añadida
            print("\nInvocando 'list_tasks' de nuevo...")
            tasks_res = await session.call_tool("list_tasks", {})
            print(f"Resultado actualizado del servidor:\n{tasks_res.content[0].text}")
            
except Exception as e:
    print("\n[ERROR] No se pudo conectar a la Azure Function.")
    print("Por favor, asegúrate de:")
    print("1. Tener instalado Azure Functions Core Tools (func)")
    print("2. Ejecutar 'func start' dentro del directorio './azure_function/'")
    print(f"Detalles del error: {e}")